# Turn HTML reference material into auditable training pairs

Parse a synthetic HTML handbook, retain source provenance, reject unsupported answers, deduplicate records, and split by source document so related answers cannot leak across train and validation.

**Audience:** Python developers building and evaluating enterprise AI applications. Familiarity with dictionaries, functions and JSON is assumed.

**Learning objectives:** extract a documented subset of HTML without script/style content; validate each answer against bounded source offsets; deduplicate QA pairs and keep each source document in one data split.

**Prerequisites:** Understand Python dictionaries, JSON, and basic HTML tags. The tutorial uses the standard-library HTMLParser and invented in-memory sources; no crawler, model, or external dataset is required.

**Setup:** use patched Python 3.12 (3.12.16 or newer) and the audited environment in the [collection README](../../README.md). Run cells in order in a fresh kernel. All default data is newly invented for this tutorial. No customer records, private repositories, credentials, downloads, or paid calls are required. Optional integrations are disabled in their own cells and never enabled by an environment variable alone.

**Execution boundary:** the results below describe the local computation shown in this notebook. They do not establish hosted-model quality or production integration success.



Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../AI_DISCLOSURE.md), [license](../../LICENSE), and [attribution](../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Create a synthetic source collection

Each document has one policy topic. Script and style text is deliberately present to test extraction. This is a bounded HTML subset example, not a web crawler or a general PDF parser.


In [ ]:
import hashlib
import json
from html.parser import HTMLParser

SOURCES = {
    "loans": "<h1>Equipment loans</h1><p>Loan duration: 14 days.</p><p>Renewal limit: 1 renewal.</p><script>private_script_text()</script>",
    "safety": "<h1>Workshop safety</h1><p>Induction duration: 45 minutes.</p><p>Required protection: safety goggles.</p><style>.hidden {color:red}</style>",
    "studio": "<h1>Ceramics studio</h1><p>Opening time: 09:00.</p><p>Closing time: 16:00.</p>",
    "refunds": "<h1>Workshop refunds</h1><p>Cancellation notice: 48 hours.</p><p>Refund method: original payment method.</p>",
}


class HandbookParser(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.skip = 0
        self.active = None
        self.buffer = []
        self.heading = ""
        self.paragraphs = []

    def handle_starttag(self, tag, attrs):
        if tag in {"script", "style"}:
            self.skip += 1
        elif not self.skip and tag in {"h1", "p"}:
            self.active = tag
            self.buffer = []

    def handle_data(self, data):
        if not self.skip and self.active:
            self.buffer.append(data)

    def handle_endtag(self, tag):
        if tag in {"script", "style"}:
            self.skip = max(0, self.skip - 1)
            return
        if tag == self.active:
            text = " ".join("".join(self.buffer).split())
            if tag == "h1":
                self.heading = text
            elif text:
                self.paragraphs.append((self.heading, text))
            self.active = None

## 2. Extract stable chunks and candidate QA

Only explicit label:value paragraphs become candidates. This deterministic rule produces narrow extractive questions. It does not invent questions with an LLM, and it records exact answer offsets in each normalized source chunk.


In [ ]:
chunks = []
for doc, html in SOURCES.items():
    parser = HandbookParser()
    parser.feed(html)
    for heading, text in parser.paragraphs:
        chunk_id = hashlib.sha256((doc + "\n" + text).encode()).hexdigest()[:16]
        chunks.append({"document": doc, "chunk_id": chunk_id, "heading": heading, "text": text})
assert len(chunks) == 8
assert not any("private_script_text" in c["text"] or ".hidden" in c["text"] for c in chunks)
candidates = []
for chunk in chunks:
    label, value = chunk["text"].split(": ", 1)
    value = value.rstrip(".")
    start = chunk["text"].index(value)
    candidates.append(
        {
            "question": f"For {chunk['heading'].lower()}, what is the {label.lower()}?",
            "answer": value,
            "source_document": chunk["document"],
            "source_chunk": chunk["chunk_id"],
            "answer_start": start,
            "answer_end": start + len(value),
        }
    )
print(candidates[0])

## 3. Validate provenance and reject a corrupted answer

Validation checks the source identifier, source-document relationship, and the exact answer span using bounded, zero-based integer offsets. Negative Python slice indices are not valid provenance offsets. It catches fabricated answers even when the row looks plausible. This is an evidence gate; a human still reviews question quality.


In [ ]:
chunk_index = {c["chunk_id"]: c for c in chunks}


def validate_pair(row):
    required = {
        "question",
        "answer",
        "source_document",
        "source_chunk",
        "answer_start",
        "answer_end",
    }
    if not isinstance(row, dict) or set(row) != required:
        raise ValueError("Unexpected QA fields")
    if not isinstance(row["source_chunk"], str) or row["source_chunk"] not in chunk_index:
        raise ValueError("Unknown source chunk")
    c = chunk_index[row["source_chunk"]]
    if row["source_document"] != c["document"]:
        raise ValueError("Source document mismatch")
    if (
        not isinstance(row["question"], str)
        or not row["question"].endswith("?")
        or not isinstance(row["answer"], str)
        or not row["answer"].strip()
    ):
        raise ValueError("Incomplete pair")
    start, end = row["answer_start"], row["answer_end"]
    if type(start) is not int or type(end) is not int or not 0 <= start < end <= len(c["text"]):
        raise ValueError("Expected bounded zero-based answer offsets")
    if c["text"][start:end] != row["answer"]:
        raise ValueError("Answer lacks exact source support")
    return row


validated = [validate_pair(r) for r in candidates]
corrupt = dict(validated[0], answer="999 days")
try:
    validate_pair(corrupt)
except ValueError:
    pass
else:
    raise AssertionError("Unsupported answer was accepted")
print({"validated": len(validated), "unsupported_rejected": True})
# Python slices accept negative and oversized indices; provenance offsets must not.
original = validated[0]
source_length = len(chunk_index[original["source_chunk"]]["text"])
invalid_spans = [
    dict(
        original,
        answer_start=original["answer_start"] - source_length,
        answer_end=original["answer_end"] - source_length,
    ),
    dict(original, answer_start=True),
    dict(original, answer_start=original["answer_end"], answer_end=original["answer_start"]),
    dict(original, answer_end=source_length + 1),
]
for invalid in invalid_spans:
    try:
        validate_pair(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid source offsets accepted")
print({"invalid_spans_rejected": len(invalid_spans)})

## 4. Deduplicate and split by document

Duplicate question-answer content is removed before splitting. Document grouping prevents two paraphrases of the same source policy from straddling the split. Four source documents are too few for reliable model selection; the mechanics are the lesson.


In [ ]:
unique = {}
for row in validated + [validated[0]]:
    fingerprint = hashlib.sha256(
        (row["question"].casefold() + "\n" + row["answer"].casefold()).encode()
    ).hexdigest()
    unique.setdefault(fingerprint, row)
ordered_docs = sorted(SOURCES, key=lambda d: hashlib.sha256(d.encode()).hexdigest())
validation_docs = set(ordered_docs[-1:])
train = [r for r in unique.values() if r["source_document"] not in validation_docs]
validation = [r for r in unique.values() if r["source_document"] in validation_docs]
assert len(unique) == 8 and len(train) == 6 and len(validation) == 2
assert {r["source_document"] for r in train}.isdisjoint(r["source_document"] for r in validation)


def chat_row(r):
    return {
        "messages": [
            {"role": "user", "content": r["question"]},
            {"role": "assistant", "content": r["answer"]},
        ]
    }


train_jsonl = "\n".join(json.dumps(chat_row(r), ensure_ascii=False) for r in train) + "\n"
assert len([json.loads(line) for line in train_jsonl.splitlines()]) == 6
print(
    {
        "train_rows": len(train),
        "validation_rows": len(validation),
        "train_sha256": hashlib.sha256(train_jsonl.encode()).hexdigest(),
    }
)

## Expected local results

The parser produces eight supported QA pairs. Deduplication leaves eight rows, split into six training and two validation rows with no shared source documents. The corrupted answer is rejected.

**Try next:** add a second paragraph wording for the same policy and decide whether exact hashing, semantic deduplication, or document grouping should handle it.


## Interpretation, limitations, and cleanup

The parser intentionally supports headings and paragraphs, not arbitrary malformed HTML. Exact spans protect factual support but cannot establish whether a question is useful or whether source material is licensed for training. Preserve provenance in a sidecar rather than assuming a training API accepts arbitrary extra columns. No training job or hosted dataset was created.

The default run keeps its artifacts in memory. Restart the kernel to release them. If you opt into a remote integration, review and remove its synthetic test records using that service's documented controls; this notebook does not delete remote data. Never save credential values or private provider output into notebook cells.


## References and provenance

All scenario text, values, evaluation cases and tutorial code were authored for this public example. No external dataset is bundled. Product names identify optional integrations and do not imply endorsement. API references were checked on 2026-10-01.

- [Python HTMLParser](https://docs.python.org/3/library/html.parser.html)
- [NVIDIA training dataset formats](https://docs.nvidia.com/nemo-helix/documentation/customizer-reference/tutorials/format-training-dataset/)
